# Yield Curve PCA & Nelson-Siegel Relative Value

**US Treasury curve, 2006 – present · Python, pandas, NumPy, SciPy · data: FRED**

This notebook asks three questions about the US Treasury curve:

1. **What actually moves the curve?** PCA on daily yield changes, decomposing moves into *level*, *slope* and *curvature*.
2. **Where should each yield be?** A Nelson-Siegel fit gives a smooth fair-value curve every day; the gap between actual and fitted yield (the *residual*) is a relative-value signal.
3. **Is that signal tradable?** A 2s5s10s butterfly, sized DV01-neutral (and PCA-neutral as a comparison), entered when the belly is unusually cheap or rich versus the fitted curve. Backtested with transaction costs, with a clean in-sample / out-of-sample split.

**Data:** FRED constant-maturity Treasury yields (`DGS3MO` … `DGS30`), daily. The series are downloaded on first run and cached to `data/ust_yields.csv`. The sample starts in March 2006 because the 30-year was not issued between 2002 and 2006.

**To run:** `pip install numpy pandas scipy matplotlib`, then *Run All*. The final cell prints a findings summary.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# --- Data -------------------------------------------------------------------
SERIES = {  # FRED id -> (label, maturity in years)
    "DGS3MO": ("3M", 0.25), "DGS6MO": ("6M", 0.5), "DGS1": ("1Y", 1.0),
    "DGS2": ("2Y", 2.0),    "DGS3": ("3Y", 3.0),   "DGS5": ("5Y", 5.0),
    "DGS7": ("7Y", 7.0),    "DGS10": ("10Y", 10.0), "DGS20": ("20Y", 20.0),
    "DGS30": ("30Y", 30.0),
}
START = "2006-03-01"          # 30Y re-issued Feb 2006
TRAIN_END = "2014-12-31"      # everything estimated on data up to here; 2015+ is out-of-sample
DATA_DIR = "data"
CACHE = os.path.join(DATA_DIR, "ust_yields.csv")
REFRESH = False               # True = re-download from FRED

# --- Trade parameters -------------------------------------------------------
BELLY, WINGS = "5Y", ("2Y", "10Y")
Z_WINDOW = 252                # rolling window (days) for the z-score
Z_ENTRY, Z_EXIT = 2.0, 0.5    # enter at |z| > 2, exit when |z| < 0.5
MAX_HOLD = 60                 # time stop, trading days
DV01_BELLY = 10_000           # $ per bp of risk on the belly leg
COST_BP_PER_LEG = 0.25        # bid/ask cost per leg, in bp of yield, per trade

# --- Plot style ---------------------------------------------------------------
C = ["#257de0", "#f47230", "#00b894", "#f2a600", "#ee7fac", "#008a17", "#6a4ce2", "#ec4647"]
plt.rcParams.update({
    "figure.figsize": (11, 4.2), "figure.dpi": 110, "axes.prop_cycle": plt.cycler(color=C),
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#999999",
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.8,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelcolor": "#666666",
    "xtick.color": "#666666", "ytick.color": "#666666", "text.color": "#191919",
    "lines.linewidth": 1.6, "legend.frameon": False, "font.size": 10,
})

## 2. Data: FRED constant-maturity Treasury yields

In [ ]:
def fetch_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    s = pd.read_csv(url, index_col=0, parse_dates=True, na_values=".").iloc[:, 0]
    return s.rename(series_id)

if os.path.exists(CACHE) and not REFRESH:
    raw = pd.read_csv(CACHE, index_col=0, parse_dates=True)
    print(f"Loaded cached data from {CACHE}")
else:
    print("Downloading from FRED ...")
    raw = pd.concat([fetch_fred(s) for s in SERIES], axis=1)
    os.makedirs(DATA_DIR, exist_ok=True)
    raw.to_csv(CACHE)
    print(f"Saved to {CACHE}")

LABELS = [SERIES[s][0] for s in SERIES]
TAU = np.array([SERIES[s][1] for s in SERIES])

yields = (raw[list(SERIES)].rename(columns={s: SERIES[s][0] for s in SERIES})
          .loc[START:].apply(pd.to_numeric, errors="coerce").dropna())   # % p.a.
print(f"{len(yields):,} trading days: {yields.index[0]:%d %b %Y} to {yields.index[-1]:%d %b %Y}")
yields.tail()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})
for lab in ["3M", "2Y", "10Y", "30Y"]:
    ax[0].plot(yields.index, yields[lab], label=lab)
ax[0].set_title("Treasury yields"); ax[0].set_ylabel("Yield (%)"); ax[0].legend(ncol=4, loc="upper right")

snap_dates = ["2006-12-29", "2012-07-25", "2021-03-31", "2023-07-03", str(yields.index[-1].date())]
for i, d in enumerate(snap_dates):
    dt = yields.index[yields.index.get_indexer([pd.Timestamp(d)], method="nearest")[0]]
    ax[1].plot(TAU, yields.loc[dt], marker="o", ms=4, color=C[i], label=f"{dt:%b %Y}")
ax[1].set_xscale("log"); ax[1].set_xticks(TAU); ax[1].set_xticklabels(LABELS, fontsize=8)
ax[1].set_title("Curve shapes"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

slope_2s10s = (yields["10Y"] - yields["2Y"]) * 100
fig, ax = plt.subplots(figsize=(12, 2.8))
ax.fill_between(slope_2s10s.index, slope_2s10s, 0, where=slope_2s10s < 0, color=C[7], alpha=0.35, label="Inverted")
ax.plot(slope_2s10s.index, slope_2s10s, color=C[0], lw=1.2, label="2s10s")
ax.axhline(0, color="#999999", lw=0.8); ax.set_title("2s10s slope (bp)"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## 3. PCA: what drives daily curve moves?

PCA is run on **daily changes** in yields (in bp), not levels, because levels are non-stationary. The covariance matrix is estimated on the **training sample only** (to 2014); those loadings are then held fixed and applied to the full sample, so the later PCA-weighted trade uses no future information.

In [ ]:
dy = yields.diff().dropna() * 100            # daily changes, bp
dy_train = dy.loc[:TRAIN_END]

def pca(changes):
    vals, vecs = np.linalg.eigh(changes.cov().values)
    order = np.argsort(vals)[::-1]
    return vals[order], vecs[:, order]

eigval, eigvec = pca(dy_train)
eigval_full, _ = pca(dy)

load = pd.DataFrame(eigvec[:, :3], index=LABELS, columns=["PC1 Level", "PC2 Slope", "PC3 Curvature"])
# Sign conventions: level = all yields up; slope = long end up vs short end; curvature = belly up vs wings
if load["PC1 Level"].mean() < 0:                       load["PC1 Level"] *= -1
if load["PC2 Slope"]["30Y"] < load["PC2 Slope"]["3M"]: load["PC2 Slope"] *= -1
if load["PC3 Curvature"]["5Y"] < load["PC3 Curvature"][["3M", "30Y"]].mean(): load["PC3 Curvature"] *= -1

expl_train = eigval / eigval.sum()
expl_full = eigval_full / eigval_full.sum()
pca_table = pd.DataFrame({"Train (to 2014)": expl_train[:5], "Full sample": expl_full[:5]},
                         index=[f"PC{i}" for i in range(1, 6)]) * 100
print("Variance explained by each component (%):")
display(pca_table.round(1))
print(f"First three PCs explain {expl_train[:3].sum():.1%} (train) / {expl_full[:3].sum():.1%} (full sample)")

scores = dy @ load.values                                # daily factor moves, full sample
scores.columns = load.columns

fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.6]})
ax[0].bar(pca_table.index, pca_table["Train (to 2014)"], color=C[0], width=0.6)
for i, v in enumerate(pca_table["Train (to 2014)"]):
    ax[0].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9, color="#191919")
ax[0].set_title("Variance explained"); ax[0].set_ylabel("%"); ax[0].grid(axis="x")
for i, col in enumerate(load.columns):
    ax[1].plot(TAU, load[col], marker="o", ms=4, color=C[i], label=col)
ax[1].axhline(0, color="#999999", lw=0.8)
ax[1].set_xscale("log"); ax[1].set_xticks(TAU); ax[1].set_xticklabels(LABELS, fontsize=8)
ax[1].set_title("PCA loadings by maturity"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Nelson-Siegel fair-value curve

$$y(\tau) = \beta_0 + \beta_1 \frac{1-e^{-\lambda\tau}}{\lambda\tau} + \beta_2 \left(\frac{1-e^{-\lambda\tau}}{\lambda\tau} - e^{-\lambda\tau}\right)$$

- $\beta_0$ = long-run **level**, $\beta_1$ = **slope** (negative when the curve slopes upward), $\beta_2$ = **curvature** (hump in the belly).
- Following Diebold & Li (2006), $\lambda$ is held **fixed**, which makes each daily fit a simple least-squares regression. $\lambda$ is chosen once by minimising fit error over the training sample (SciPy), then frozen.

In [ ]:
def ns_basis(tau, lam):
    x = lam * tau
    f1 = (1 - np.exp(-x)) / x
    return np.column_stack([np.ones_like(tau), f1, f1 - np.exp(-x)])

def fit_ns(Y, lam):
    """Y: DataFrame of yields (days x maturities). Returns betas, fitted yields."""
    X = ns_basis(TAU, lam)
    betas, *_ = np.linalg.lstsq(X, Y.values.T, rcond=None)      # 3 x T
    fitted = pd.DataFrame((X @ betas).T, index=Y.index, columns=Y.columns)
    return pd.DataFrame(betas.T, index=Y.index, columns=["beta0", "beta1", "beta2"]), fitted

def train_rmse(lam):
    _, f = fit_ns(yields.loc[:TRAIN_END], lam)
    return np.sqrt(((yields.loc[:TRAIN_END] - f) ** 2).values.mean())

opt = minimize_scalar(train_rmse, bounds=(0.05, 3.0), method="bounded")
LAM = opt.x
print(f"Optimal lambda = {LAM:.3f} per year  -> curvature loading peaks at {1.7932 / LAM:.1f}y maturity")

betas, fitted = fit_ns(yields, LAM)
resid = (yields - fitted) * 100                                  # bp; positive = yield ABOVE fair value = cheap
rmse_bp = np.sqrt((resid ** 2).mean())
print(f"Overall fit RMSE: {np.sqrt((resid ** 2).values.mean()):.1f} bp")

# Link NS factors to PCA factors: correlation of daily changes
dbeta = betas.diff().dropna()
link = pd.Series({
    "Level:     d(beta0)  vs PC1": dbeta["beta0"].corr(scores["PC1 Level"]),
    "Slope:    -d(beta1)  vs PC2": (-dbeta["beta1"]).corr(scores["PC2 Slope"]),
    "Curvature: d(beta2)  vs PC3": dbeta["beta2"].corr(scores["PC3 Curvature"]),
}, name="corr")
print("\nNS factors vs PCA factors (daily changes):"); display(link.to_frame().round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
ax[0].bar(LABELS, rmse_bp, color=C[0], width=0.6); ax[0].grid(axis="x")
ax[0].set_title("Fit error by maturity (RMSE, bp)"); ax[0].tick_params(axis="x", labelsize=8)
dt = yields.index[-1]
ax[1].plot(TAU, yields.loc[dt], "o", ms=6, color=C[0], label="Actual")
tt = np.linspace(0.25, 30, 200)
ax[1].plot(tt, ns_basis(tt, LAM) @ betas.loc[dt].values, color=C[1], label="Nelson-Siegel fit")
ax[1].set_xscale("log"); ax[1].set_xticks(TAU); ax[1].set_xticklabels(LABELS, fontsize=8)
ax[1].set_title(f"Fit on {dt:%d %b %Y}"); ax[1].set_ylabel("Yield (%)"); ax[1].legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(3, 1, figsize=(12, 6.5), sharex=True)
for i, (col, name) in enumerate([("beta0", "Level  (beta0)"), ("beta1", "Slope  (beta1)"), ("beta2", "Curvature  (beta2)")]):
    ax[i].plot(betas.index, betas[col], color=C[i], lw=1.2); ax[i].set_title(name, loc="left", fontsize=10)
    ax[i].axhline(0, color="#999999", lw=0.8)
plt.tight_layout(); plt.show()

## 5. Residuals: are mispricings mean-reverting?

If the residual (actual − fitted) is a genuine relative-value signal, it should be **small** and **mean-reverting**. Half-life comes from an AR(1) fit, $\Delta r_t = a + b\,r_{t-1} + \varepsilon_t$, with half-life $= -\ln 2 / \ln(1+b)$.

In [ ]:
def half_life(x):
    x = x.dropna()
    lag, d = x.shift(1).iloc[1:], x.diff().iloc[1:]
    b = np.polyfit(lag.values, d.values, 1)[0]
    return np.inf if b >= 0 else -np.log(2) / np.log(1 + b)

res_stats = pd.DataFrame({
    "Mean (bp)": resid.mean(), "Std (bp)": resid.std(), "Mean |resid| (bp)": resid.abs().mean(),
    "Half-life (days)": resid.apply(half_life),
})
display(res_stats.round(1))

fig, ax = plt.subplots(figsize=(12, 3.4))
for i, lab in enumerate(["2Y", "5Y", "10Y"]):
    ax.plot(resid.index, resid[lab], lw=1, color=C[i], label=lab)
ax.axhline(0, color="#999999", lw=0.8)
ax.set_title("Nelson-Siegel residuals (bp; positive = cheap)"); ax.legend(ncol=3)
plt.tight_layout(); plt.show()

## 6. Building the 2s5s10s butterfly

Long the belly (5Y) against the wings (2Y, 10Y). Weights are expressed **in DV01 terms**, so a weight of −0.5 on the 2Y means half the belly's dollar risk per bp, sold.

| Construction | Weights (2Y / 5Y / 10Y) | Hedges |
|---|---|---|
| **DV01 50/50** | −0.5 / +1 / −0.5 | Parallel shifts only |
| **PCA-neutral** | solved from training loadings | Level *and* slope (PC1 and PC2) |

The **fly yield** is $w \cdot y$. A long fly (+1) earns $-\Delta(\text{fly}) \times \text{DV01}$: it profits when the belly richens versus the wings.

In [ ]:
legs = [WINGS[0], BELLY, WINGS[1]]

w_dv01 = pd.Series({WINGS[0]: -0.5, BELLY: 1.0, WINGS[1]: -0.5})

# PCA-neutral: belly = 1, solve wing weights so the fly has zero exposure to PC1 and PC2
L = load.loc[legs, ["PC1 Level", "PC2 Slope"]].values            # 3 x 2
A = L[[0, 2], :].T                                               # unknowns: wing weights
b = -L[1, :]                                                     # belly contribution
w2, w10 = np.linalg.solve(A, b)
w_pca = pd.Series({WINGS[0]: w2, BELLY: 1.0, WINGS[1]: w10})

weights = pd.DataFrame({"DV01 50/50": w_dv01, "PCA-neutral": w_pca}).loc[legs]
display(weights.round(3))

# Factor exposure of each fly: regress daily fly changes on PC scores
def exposures(w, sample):
    f = (dy[legs] @ w[legs]).loc[sample]
    X = np.column_stack([np.ones(len(f)), scores.loc[f.index].values])
    beta = np.linalg.lstsq(X, f.values, rcond=None)[0][1:]
    return pd.Series(beta, index=["Level beta", "Slope beta", "Curv beta"])

expo = pd.DataFrame({f"{n} ({lab})": exposures(weights[n], s)
                     for n in weights.columns
                     for lab, s in [("train", slice(None, TRAIN_END)), ("test", slice(TRAIN_END, None))]}).T
print("Sensitivity of daily fly change (bp) to a 1-unit move in each PCA factor:")
display(expo.round(3))

### What the DV01 sizing means in actual bonds

Using par-bond modified duration, $D = \frac{1 - (1+y/2)^{-2T}}{y}$, this converts the DV01 weights into face values for the latest date.

In [ ]:
def par_mod_duration(y_pct, T):
    y = y_pct / 100
    return T if y < 1e-6 else (1 - (1 + y / 2) ** (-2 * T)) / y

last = yields.index[-1]
rows = []
for lab in legs:
    T = TAU[LABELS.index(lab)]; y = yields.loc[last, lab]; D = par_mod_duration(y, T)
    dv01_per_mm = D * 1e6 * 1e-4                     # $ per bp per $1mm face (par bond)
    row = {"Yield (%)": y, "Mod. duration": D, "DV01 per $1mm": dv01_per_mm}
    for n in weights.columns:
        row[f"Face ($mm), {n}"] = weights.loc[lab, n] * DV01_BELLY / dv01_per_mm
    rows.append(pd.Series(row, name=lab))
sizing = pd.DataFrame(rows)
print(f"Sizing on {last:%d %b %Y} for ${DV01_BELLY:,} DV01 on the belly (negative = short):")
display(sizing.round(2))

## 7. Signal and backtest

- **Signal:** fly residual $s_t = w \cdot \text{resid}_t$ (bp vs Nelson-Siegel fair value), converted to a rolling 252-day z-score.
- **Rules:** $z > +2$ → belly cheap → **buy the fly**. $z < -2$ → belly rich → **sell the fly**. Exit when $|z| < 0.5$ or after 60 trading days.
- **No look-ahead:** positions are decided at the close on day $t$ and earn the P&L from $t$ to $t+1$.
- **Costs:** 0.25 bp per leg each time the position changes, scaled by the leg's DV01 weight.
- **Baseline:** the same rules on a z-score of the *raw* fly yield, to test whether the Nelson-Siegel fit adds anything over a naive spread.

In [ ]:
def make_positions(z, entry, exit_, max_hold):
    pos, cur, held = np.zeros(len(z)), 0, 0
    for i, zi in enumerate(z.values):
        if np.isnan(zi):
            cur, held = 0, 0
        elif cur == 0:
            if zi > entry:   cur, held = 1, 0
            elif zi < -entry: cur, held = -1, 0
        else:
            held += 1
            if (cur == 1 and zi < exit_) or (cur == -1 and zi > -exit_) or held >= max_hold:
                cur = 0
        pos[i] = cur
    return pd.Series(pos, index=z.index)

def zscore(s, window=Z_WINDOW):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

def backtest(w, signal="ns", entry=Z_ENTRY, exit_=Z_EXIT, max_hold=MAX_HOLD, cost=COST_BP_PER_LEG):
    fly = (yields[legs] * 100) @ w[legs]                       # fly yield, bp
    raw_sig = (resid[legs] @ w[legs]) if signal == "ns" else fly
    z = zscore(raw_sig)
    pos = make_positions(z, entry, exit_, max_hold)
    gross = -pos.shift(1).fillna(0) * fly.diff().fillna(0) * DV01_BELLY
    turnover = pos.diff().abs().fillna(pos.abs())
    costs = turnover * cost * w[legs].abs().sum() * DV01_BELLY
    return pd.DataFrame({"fly": fly, "signal": raw_sig, "z": z, "pos": pos,
                         "gross": gross, "cost": costs, "pnl": gross - costs,
                         "unit_cost": cost * w[legs].abs().sum() * DV01_BELLY})

def trade_list(bt):
    """One row per trade: holding days, side, P&L net of a round-trip cost."""
    pos = bt["pos"]
    new = (pos != 0) & (pos != pos.shift(1, fill_value=0))
    tid = new.cumsum().where(pos != 0)                          # trade id while the position is on
    gross = bt["gross"].groupby(tid.shift(1)).sum()             # day t P&L belongs to position held at close t-1
    t = pd.DataFrame({"days": tid.value_counts().sort_index(),
                      "side": pos.groupby(tid).first()})
    entry_cost = bt["unit_cost"].where(new).groupby(tid).first()
    t["pnl"] = gross.reindex(t.index).fillna(0) - 2 * entry_cost.reindex(t.index)
    return t

def stats(bt, sample=slice(None)):
    p = bt["pnl"].loc[sample]; b = bt.loc[sample]
    cum = p.cumsum(); dd = cum - cum.cummax()
    tr = trade_list(b) if (b["pos"] != 0).any() else pd.DataFrame(columns=["pnl", "days"])
    return pd.Series({
        "Total P&L ($)": p.sum(),
        "Ann. P&L ($)": p.mean() * 252,
        "Sharpe": p.mean() / p.std() * np.sqrt(252) if p.std() > 0 else np.nan,
        "Max drawdown ($)": dd.min(),
        "Trades": len(tr),
        "Hit rate": (tr["pnl"] > 0).mean() if len(tr) else np.nan,
        "Avg hold (days)": tr["days"].mean() if len(tr) else np.nan,
        "Time in market": (b["pos"] != 0).mean(),
        "Costs ($)": b["cost"].sum(),
    })

variants = {
    "NS signal, DV01 50/50":  backtest(w_dv01, "ns"),
    "NS signal, PCA-neutral": backtest(w_pca, "ns"),
    "Raw fly, DV01 50/50":    backtest(w_dv01, "raw"),
}
samples = {"Train 2006-14": slice(None, TRAIN_END), "Test 2015+": slice("2015-01-01", None),
           "2022 only": slice("2022-01-01", "2022-12-31")}
results = pd.concat({(v, s): stats(bt, smp) for v, bt in variants.items() for s, smp in samples.items()}, axis=1).T
fmt = results.copy()
for c in ["Total P&L ($)", "Ann. P&L ($)", "Max drawdown ($)", "Costs ($)"]:
    fmt[c] = fmt[c].map(lambda x: f"{x:,.0f}")
for c in ["Hit rate", "Time in market"]:
    fmt[c] = fmt[c].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
fmt["Sharpe"] = fmt["Sharpe"].map(lambda x: f"{x:.2f}")
fmt["Avg hold (days)"] = fmt["Avg hold (days)"].map(lambda x: f"{x:.0f}" if pd.notna(x) else "-")
fmt["Trades"] = fmt["Trades"].astype(int)
display(fmt)

In [ ]:
main = variants["NS signal, PCA-neutral"]
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})
ax[0].plot(main.index, main["z"], color=C[0], lw=0.9, label="Fly residual z-score")
for lvl in [Z_ENTRY, -Z_ENTRY]:
    ax[0].axhline(lvl, color=C[7], lw=0.9, ls="--")
ax[0].fill_between(main.index, -6, 6, where=main["pos"] > 0, color=C[2], alpha=0.15, label="Long fly")
ax[0].fill_between(main.index, -6, 6, where=main["pos"] < 0, color=C[1], alpha=0.15, label="Short fly")
ax[0].set_ylim(-5, 5); ax[0].set_title("Signal and positions (NS, PCA-neutral)"); ax[0].legend(ncol=3, loc="upper left")

for i, (name, bt) in enumerate(variants.items()):
    ax[1].plot(bt.index, bt["pnl"].cumsum() / 1e3, color=C[i], label=name)
ax[1].axvline(pd.Timestamp(TRAIN_END), color="#666666", lw=1, ls=":")
ax[1].text(pd.Timestamp(TRAIN_END), ax[1].get_ylim()[1] * 0.9, "  out-of-sample ->", color="#666666", fontsize=9)
ax[1].axhline(0, color="#999999", lw=0.8)
ax[1].set_title(f"Cumulative net P&L (\\$k, \\${DV01_BELLY:,} belly DV01)"); ax[1].set_ylabel("\\$k"); ax[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

annual = pd.DataFrame({n: bt["pnl"].groupby(bt.index.year).sum() / 1e3 for n, bt in variants.items()})
fig, ax = plt.subplots(figsize=(12, 3.4))
xs = np.arange(len(annual)); wbar = 0.27
for i, n in enumerate(annual.columns):
    ax.bar(xs + (i - 1) * wbar, annual[n], width=wbar - 0.02, color=C[i], label=n)
ax.set_xticks(xs); ax.set_xticklabels(annual.index, fontsize=8); ax.axhline(0, color="#999999", lw=0.8)
ax.set_title("Net P&L by calendar year (\\$k)"); ax.legend(ncol=3, fontsize=8); ax.grid(axis="x")
plt.tight_layout(); plt.show()

## 8. Robustness: sensitivity to the entry and exit thresholds

A result that only works at one parameter setting is overfit. This grid reruns the main strategy (NS signal, PCA-neutral) across thresholds and reports **out-of-sample** Sharpe.

In [ ]:
grid = pd.DataFrame(index=[1.5, 2.0, 2.5], columns=[0.0, 0.5, 1.0], dtype=float)
for e in grid.index:
    for x in grid.columns:
        grid.loc[e, x] = stats(backtest(w_pca, "ns", entry=e, exit_=x), samples["Test 2015+"])["Sharpe"]
grid.index.name, grid.columns.name = "Entry |z|", "Exit |z|"
print("Out-of-sample Sharpe (2015+), NS signal, PCA-neutral fly:")
display(grid.round(2))

cost_sens = pd.Series({c: stats(backtest(w_pca, "ns", cost=c), samples["Test 2015+"])["Sharpe"]
                       for c in [0.0, 0.25, 0.5, 1.0]}, name="OOS Sharpe")
cost_sens.index.name = "Cost per leg (bp)"
display(cost_sens.round(2).to_frame())

## 9. Version 2: fixing what v1 got wrong

v1 found no edge, and its diagnostics pointed to four specific weaknesses. v2 fixes each one and adds them **one at a time**, so the contribution of each fix is visible.

| # | v1 weakness (diagnosed above) | v2 fix |
|---|---|---|
| 1 | P&L ignored **carry and roll-down**, which a real position earns or pays every day | Add daily carry (yield minus 3M repo proxy) and roll-down along the fitted Nelson-Siegel curve |
| 2 | PCA hedge estimated once; it **leaked level and slope** exposure out of sample | Re-estimate PCA on a trailing 2-year window every month |
| 3 | With a 40+ day half-life, trades often **paid carry while waiting** for reversion | Only open a trade if its carry + roll is in our favour |
| 4 | The **20Y** showed the largest, most persistent dislocation | Run the same strategy on a 10s20s30s fly |

**Research-integrity note.** These changes were chosen from v1's *diagnostics* (exposures, half-lives, missing P&L components), not by searching for profitable settings. Entry/exit thresholds, costs and the z-score window are **unchanged**. Because v1 was already run on 2015+, that period is no longer a pristine hold-out for v2, so v2 results should be read with that caveat.

In [ ]:
# --- Fix 1: daily carry and roll-down per leg (bp of yield per day, per 1 unit of DV01) ---
DT = 1 / 252
TAU_OF = dict(zip(LABELS, TAU))

def par_mod_duration_vec(y_pct, T):
    y = np.asarray(y_pct, dtype=float) / 100
    with np.errstate(divide="ignore", invalid="ignore"):
        d = (1 - (1 + y / 2) ** (-2 * T)) / y
    return np.where(y < 1e-6, T, d)

carry_bp, roll_bp = {}, {}
for lab in LABELS[1:]:                                     # 3M is the funding proxy
    T = TAU_OF[lab]
    D = par_mod_duration_vec(yields[lab], T)
    carry_bp[lab] = (yields[lab] - yields["3M"]) * 100 / (252 * D)
    on_curve = betas.values @ ns_basis(np.array([T, T - DT]), LAM).T   # fitted yield at T and T - 1 day
    roll_bp[lab] = pd.Series((on_curve[:, 0] - on_curve[:, 1]) * 100, index=yields.index)
carry_bp, roll_bp = pd.DataFrame(carry_bp), pd.DataFrame(roll_bp)
carry_roll = carry_bp + roll_bp

print("Average carry + roll-down, bp per year of yield-equivalent (long 1 unit of DV01):")
display((carry_roll * 252).mean().round(1).to_frame("bp / yr").T)

In [ ]:
# --- Fix 2: rolling PCA hedge weights (no look-ahead: weights dated at the last day of their window) ---
PCA_WINDOW, PCA_STEP = 504, 21

def pca_neutral_weights(load_df, fly_legs):
    L = load_df.loc[fly_legs].values[:, :2]
    a, c = np.linalg.solve(L[[0, 2], :].T, -L[1, :])
    return pd.Series({fly_legs[0]: a, fly_legs[1]: 1.0, fly_legs[2]: c})

MAX_WING = 3.0   # guard: if near-collinear legs give an extreme hedge, keep the previous month's weights

def rolling_pca_weights(fly_legs, window=PCA_WINDOW, step=PCA_STEP):
    rows, last, rejected = {}, None, 0
    for i in range(window, len(dy) + 1, step):
        _, vecs = pca(dy.iloc[i - window:i])
        w = pca_neutral_weights(pd.DataFrame(vecs, index=LABELS), fly_legs)
        if not np.isfinite(w).all() or w.abs().max() > MAX_WING:
            rejected += 1
            if last is None:
                continue
            w = last
        rows[dy.index[i - 1]], last = w, w
    print(f"{'/'.join(fly_legs)}: {len(rows)} monthly re-estimates, {rejected} rejected as unstable")
    return pd.DataFrame(rows).T.reindex(yields.index).ffill()

FLIES = {"2s5s10s": ["2Y", "5Y", "10Y"], "10s20s30s": ["10Y", "20Y", "30Y"]}
W_static = {f: pca_neutral_weights(load, l) for f, l in FLIES.items()}
W_roll = {f: rolling_pca_weights(l) for f, l in FLIES.items()}

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), sharey=False)
for k, (f, l) in enumerate(FLIES.items()):
    for j, leg in enumerate([l[0], l[2]]):
        ax[k].plot(W_roll[f].index, W_roll[f][leg], color=C[j], lw=1.2, label=f"{leg} (rolling)")
        ax[k].axhline(W_static[f][leg], color=C[j], ls="--", lw=1, label=f"{leg} (static, train)")
    ax[k].set_title(f"{f}: wing weights per 1.0 of belly DV01"); ax[k].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

In [ ]:
# --- v2 backtest engine: time-varying weights, optional carry, optional carry filter ---
def make_positions_v2(z, entry, exit_, max_hold, allow_long, allow_short):
    pos, cur, held = np.zeros(len(z)), 0, 0
    for i, zi in enumerate(z.values):
        if np.isnan(zi):
            cur, held = 0, 0
        elif cur == 0:
            if zi > entry and allow_long[i]:     cur, held = 1, 0
            elif zi < -entry and allow_short[i]: cur, held = -1, 0
        else:
            held += 1
            if (cur == 1 and zi < exit_) or (cur == -1 and zi > -exit_) or held >= max_hold:
                cur = 0
        pos[i] = cur
    return pd.Series(pos, index=z.index)

def backtest_v2(fly_legs, W, carry=False, carry_filter=False, entry=Z_ENTRY, exit_=Z_EXIT,
                max_hold=MAX_HOLD, cost=COST_BP_PER_LEG):
    if isinstance(W, pd.Series):
        W = pd.DataFrame([W] * len(yields), index=yields.index)
    W = W[fly_legs]
    n = len(fly_legs)
    signal = (resid[fly_legs] * W).sum(axis=1, min_count=n)            # NS residual of the fly, bp
    z = zscore(signal)
    cr = (carry_roll[fly_legs] * W).sum(axis=1, min_count=n).fillna(0)  # long-fly carry+roll, bp/day
    ok_long = (cr > 0).values if carry_filter else np.ones(len(z), bool)
    ok_short = (cr < 0).values if carry_filter else np.ones(len(z), bool)
    pos = make_positions_v2(z, entry, exit_, max_hold, ok_long, ok_short)
    held = pos.shift(1).fillna(0)
    move = (yields[fly_legs].diff() * 100 * W.shift(1)).sum(axis=1).fillna(0)  # fly move at yesterday's weights
    price_pnl = -held * move * DV01_BELLY
    carry_pnl = held * cr.shift(1).fillna(0) * DV01_BELLY if carry else 0 * held
    unit = cost * W.abs().sum(axis=1).fillna(0) * DV01_BELLY
    costs = pos.diff().abs().fillna(pos.abs()) * unit
    gross = price_pnl + carry_pnl
    return pd.DataFrame({"signal": signal, "z": z, "pos": pos, "price_pnl": price_pnl, "carry_pnl": carry_pnl,
                         "gross": gross, "cost": costs, "pnl": gross - costs, "unit_cost": unit})

STEPS = {
    "v1 baseline (static PCA, no carry)": dict(W="static", carry=False, carry_filter=False),
    "+ carry & roll in P&L":              dict(W="static", carry=True,  carry_filter=False),
    "+ rolling PCA hedge":                dict(W="roll",   carry=True,  carry_filter=False),
    "+ carry filter (v2)":                dict(W="roll",   carry=True,  carry_filter=True),
}
v2_bt = {}
for f, l in FLIES.items():
    for step, kw in STEPS.items():
        W = W_static[f] if kw["W"] == "static" else W_roll[f]
        v2_bt[(f, step)] = backtest_v2(l, W, carry=kw["carry"], carry_filter=kw["carry_filter"])

# Sanity check: the v2 engine reproduces v1 for the 2s5s10s PCA fly
assert np.isclose(v2_bt[("2s5s10s", "v1 baseline (static PCA, no carry)")]["pnl"].sum(),
                  variants["NS signal, PCA-neutral"]["pnl"].sum()), "v2 engine does not reproduce v1"

v2_results = pd.concat({(f, step, s): stats(bt, smp)
                        for (f, step), bt in v2_bt.items() for s, smp in samples.items()}, axis=1).T

def fmt_table(r):
    r = r.copy()
    for c in ["Total P&L ($)", "Ann. P&L ($)", "Max drawdown ($)", "Costs ($)"]:
        r[c] = r[c].map(lambda x: f"{x:,.0f}")
    for c in ["Hit rate", "Time in market"]:
        r[c] = r[c].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
    r["Sharpe"] = r["Sharpe"].map(lambda x: f"{x:.2f}")
    r["Avg hold (days)"] = r["Avg hold (days)"].map(lambda x: f"{x:.0f}" if pd.notna(x) else "-")
    r["Trades"] = r["Trades"].astype(int)
    return r

v2_fmt = fmt_table(v2_results)
display(v2_fmt[["Total P&L ($)", "Sharpe", "Max drawdown ($)", "Trades", "Hit rate", "Avg hold (days)", "Costs ($)"]])

In [ ]:
# Did the rolling hedge actually remove factor exposure out of sample?
def fly_exposure(fly_legs, W, sample):
    if isinstance(W, pd.Series):
        W = pd.DataFrame([W] * len(yields), index=yields.index)
    move = (dy[fly_legs] * W[fly_legs].shift(1).reindex(dy.index)).sum(axis=1, min_count=3).loc[sample].dropna()
    X = np.column_stack([np.ones(len(move)), scores.loc[move.index].values])
    return pd.Series(np.linalg.lstsq(X, move.values, rcond=None)[0][1:], index=["Level beta", "Slope beta", "Curv beta"])

test = slice("2015-01-01", None)
v2_expo = pd.DataFrame({f"{f}, {kind}": fly_exposure(l, W[f], test)
                        for f, l in FLIES.items() for kind, W in [("static PCA", W_static), ("rolling PCA", W_roll)]}).T
print("Out-of-sample (2015+) factor exposure of each fly:")
display(v2_expo.round(3))

# Where does v2's P&L come from? (2015+)
decomp = pd.DataFrame({f: v2_bt[(f, "+ carry filter (v2)")].loc[test, ["price_pnl", "carry_pnl", "cost", "pnl"]].sum()
                       for f in FLIES}).T
decomp.columns = ["Price P&L ($)", "Carry + roll ($)", "Costs ($)", "Net ($)"]
print("v2 P&L decomposition, 2015+:")
display(decomp.round(0))

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for k, f in enumerate(FLIES):
    for i, step in enumerate(STEPS):
        bt = v2_bt[(f, step)]
        ax[k].plot(bt.index, bt["pnl"].cumsum() / 1e3, color=C[i], lw=1.3, label=step)
    ax[k].axvline(pd.Timestamp(TRAIN_END), color="#666666", lw=1, ls=":")
    ax[k].axhline(0, color="#999999", lw=0.8)
    ax[k].set_title(f"{f}: cumulative net P&L (\\$k)"); ax[k].legend(fontsize=7.5, loc="upper left")
plt.tight_layout(); plt.show()

v2_grid = {}
for f, l in FLIES.items():
    g = pd.DataFrame(index=[1.5, 2.0, 2.5], columns=[0.0, 0.5, 1.0], dtype=float)
    for e in g.index:
        for x in g.columns:
            g.loc[e, x] = stats(backtest_v2(l, W_roll[f], carry=True, carry_filter=True, entry=e, exit_=x), test)["Sharpe"]
    g.index.name, g.columns.name = "Entry |z|", "Exit |z|"
    v2_grid[f] = g
    print(f"v2 out-of-sample Sharpe grid, {f}:"); display(g.round(2))

## 10. Findings summary

Copy the output of this cell to write up the results (v1 first, then v2).

In [ ]:
main_t = results.loc[("NS signal, PCA-neutral", "Test 2015+")]
main_tr = results.loc[("NS signal, PCA-neutral", "Train 2006-14")]
print(f"""
DATA        {yields.index[0]:%b %Y} - {yields.index[-1]:%b %Y}, {len(yields):,} days, {len(LABELS)} maturities (3M-30Y)

PCA (train) Level {expl_train[0]:.1%} | Slope {expl_train[1]:.1%} | Curvature {expl_train[2]:.1%} | top 3 = {expl_train[:3].sum():.1%}
PCA (full)  Level {expl_full[0]:.1%} | Slope {expl_full[1]:.1%} | Curvature {expl_full[2]:.1%}

NELSON-SIEGEL  lambda = {LAM:.3f}  | overall RMSE = {np.sqrt((resid**2).values.mean()):.1f} bp
               worst-fit maturity: {rmse_bp.idxmax()} ({rmse_bp.max():.1f} bp) | best: {rmse_bp.idxmin()} ({rmse_bp.min():.1f} bp)
               NS-PCA link (corr of daily changes): level {link.iloc[0]:.2f}, slope {link.iloc[1]:.2f}, curvature {link.iloc[2]:.2f}
               5Y residual: std {res_stats.loc['5Y','Std (bp)']:.1f} bp, half-life {res_stats.loc['5Y','Half-life (days)']:.0f} days

FLY WEIGHTS    DV01 50/50: {w_dv01.round(2).to_dict()}
               PCA-neutral: {w_pca.round(2).to_dict()}
               Slope beta, DV01 fly (test): {expo.loc['DV01 50/50 (test)','Slope beta']:.3f} | PCA fly (test): {expo.loc['PCA-neutral (test)','Slope beta']:.3f}
""")
print("BACKTEST (net of costs):")
display(fmt[["Total P&L ($)", "Sharpe", "Max drawdown ($)", "Trades", "Hit rate", "Avg hold (days)"]])
print("OOS Sharpe grid (entry x exit):"); display(grid.round(2))
print("OOS Sharpe vs cost per leg:"); display(cost_sens.round(2).to_frame().T)
print("=" * 80, "\nVERSION 2")
print("Average carry + roll (bp/yr per unit DV01):", (carry_roll * 252).mean().round(1).to_dict())
for f in FLIES:
    w = W_roll[f].dropna()
    print(f"Rolling PCA weights {f}: " + ", ".join(f"{c} {w[c].min():.2f} to {w[c].max():.2f}" for c in w.columns))
print("\nOOS factor exposure:"); display(v2_expo.round(3))
print("Step-by-step results:")
display(v2_fmt[["Total P&L ($)", "Sharpe", "Max drawdown ($)", "Trades", "Hit rate", "Avg hold (days)"]])
print("v2 P&L decomposition, 2015+:"); display(decomp.round(0))
for f in FLIES:
    print(f"v2 OOS Sharpe grid, {f}:"); display(v2_grid[f].round(2))

## Limitations

- **Constant-maturity yields, not tradable bonds.** CMT yields are interpolated; a real trade uses specific on-the-run or off-the-run issues with their own liquidity, specialness and roll.
- **Carry is approximate (v2).** Carry uses the 3M bill as a repo proxy and par-bond durations; real repo rates, specials and coupon effects differ by issue. v1 ignores carry entirely.
- **Fixed λ** keeps the fit stable but can't capture every curve shape; persistent residuals at the short end partly reflect model misfit, not mispricing.
- **Cost model is simple.** 0.25 bp per leg is reasonable for on-the-run Treasuries in normal markets, but costs widen sharply in stress (March 2020).
- **Regime dependence.** Mean reversion in the curve's shape can break during fast hiking or easing cycles, when the belly re-prices for policy expectations.

*Reference: Diebold, F. X. & Li, C. (2006), "Forecasting the term structure of government bond yields", Journal of Econometrics. Litterman, R. & Scheinkman, J. (1991), "Common factors affecting bond returns", Journal of Fixed Income.*